In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *
from MODELS.pipeline import *

### Load Model

In [ ]:
PTSmodel = joblib.load('Models/xgbPTSModelNoOppStats26.pkl')
PTSfeatures = joblib.load('Models/topPTSfeaturesNoOppStats26.pkl')

### Load Data

In [ ]:
pd.set_option('display.max_columns', None)


s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv('../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_20251010.csv')
singlePTSBookies = usData[usData['CATEGORY'] == 'player_points']

dfsData = pd.read_csv('../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_20251019.csv')
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]
dfsPTS

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Luguentz Dort,Over,10.0,-137,2025-10-21,2025-10-20T03:05:29Z
1,PrizePicks,player_points,Luguentz Dort,Under,10.0,-137,2025-10-21,2025-10-20T03:05:29Z
2,PrizePicks,player_points,Alperen Sengun,Over,18.5,-137,2025-10-21,2025-10-20T03:05:29Z
3,PrizePicks,player_points,Alperen Sengun,Under,18.5,-137,2025-10-21,2025-10-20T03:05:29Z
4,PrizePicks,player_points,Jabari Smith Jr,Over,12.0,-137,2025-10-21,2025-10-20T03:05:29Z
...,...,...,...,...,...,...,...,...
286,PrizePicks,player_points,Dario Saric,Under,8.5,-137,2025-10-23,2025-10-20T03:10:32Z
287,PrizePicks,player_points,Drew Eubanks,Over,8.5,-137,2025-10-23,2025-10-20T03:10:32Z
288,PrizePicks,player_points,Drew Eubanks,Under,8.5,-137,2025-10-23,2025-10-20T03:10:32Z
289,PrizePicks,player_points,Mark Williams,Over,12.5,-137,2025-10-23,2025-10-20T03:10:32Z


### Top EVs for single bets

In [4]:
results = single_bet(
    data=s25,
    bookmakers=singlePTSBookies,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=4.5,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
71,Shai Gilgeous-Alexander,Bovada,player_points,27.5,200,over,35.492397,1,0.938,0.062,0.333,181.34,0.91,0.45,0.23,"(25.5, 46.1)"
47,Jalen Williams,Bovada,player_points,16.5,190,over,25.813309,1,0.948,0.052,0.345,174.80,0.92,0.46,0.23,"(14.6, 37.1)"
73,Shai Gilgeous-Alexander,Bovada,player_points,28.5,165,over,35.492397,1,0.902,0.098,0.377,138.92,0.84,0.42,0.21,"(25.0, 46.1)"
49,Jalen Williams,Bovada,player_points,17.5,150,over,25.813309,1,0.928,0.072,0.400,132.00,0.88,0.44,0.22,"(14.8, 36.9)"
61,Luguentz Dort,Bovada,player_points,6.5,205,over,10.725503,0,0.760,0.240,0.328,131.89,0.64,0.32,0.16,"(0.0, 21.6)"
27,Chet Holmgren,Bovada,player_points,13.5,180,over,19.763552,1,0.745,0.255,0.357,108.60,0.60,0.30,0.15,"(1.4, 37.6)"
75,Shai Gilgeous-Alexander,Bovada,player_points,29.5,140,over,35.492397,1,0.868,0.132,0.417,108.22,0.77,0.39,0.19,"(24.9, 45.9)"
51,Jalen Williams,Bovada,player_points,18.5,120,over,25.813309,1,0.902,0.098,0.455,98.40,0.82,0.41,0.20,"(14.7, 36.8)"
58,Jalen Williams,Bovada,player_points,23.5,185,over,25.813309,0,0.656,0.344,0.351,86.93,0.47,0.23,0.12,"(14.9, 36.9)"
56,Jalen Williams,Bovada,player_points,22.5,150,over,25.813309,0,0.723,0.277,0.400,80.67,0.54,0.27,0.13,"(14.5, 37.1)"


### Top EVs for 2 leg bets

In [6]:
results = prizepickspairsEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5,
    stake=100,
    simulations=1000,
    std_window=15,
    min_std=2.0,
    max_std=8.5,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Deni Avdija,player_points,PrizePicks,-137,19.5,28.56,OVER,0.95,0.05,"(19.0, 38.4)",UNDER/OVER,0,0.7097,1.129,0.564
1,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Drew Eubanks,player_points,PrizePicks,-137,8.5,2.98,UNDER,0.05,0.95,"(0.0, 8.1)",UNDER/UNDER,0,0.7097,1.129,0.564
2,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Guerschon Yabusele,player_points,PrizePicks,-137,5.5,15.33,OVER,0.95,0.05,"(4.3, 27.1)",UNDER/OVER,0,0.7097,1.129,0.564
3,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.2)",UNDER/UNDER,0,0.7097,1.129,0.564
4,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Jaylon Tyson,player_points,PrizePicks,-137,6.5,18.19,OVER,0.95,0.05,"(9.7, 25.3)",UNDER/OVER,0,0.7097,1.129,0.564
5,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Lonzo Ball,player_points,PrizePicks,-137,7.5,12.18,OVER,0.95,0.05,"(7.2, 17.0)",UNDER/OVER,0,0.7097,1.129,0.564
6,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Mikal Bridges,player_points,PrizePicks,-137,15.5,5.20,UNDER,0.05,0.95,"(0.0, 14.7)",UNDER/UNDER,0,0.7097,1.129,0.564
7,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Trey Murphy III,player_points,PrizePicks,-137,19.5,1.70,UNDER,0.05,0.95,"(0.0, 11.8)",UNDER/UNDER,0,0.7097,1.129,0.564
8,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Stephen Curry,player_points,PrizePicks,-137,0.5,29.74,OVER,0.95,0.05,"(15.8, 44.0)",UNDER/OVER,0,0.7097,1.129,0.564
9,Alex Sarr,player_points,PrizePicks,-137,15.5,12.64,UNDER,0.253,0.747,"(1.7, 22.7)",Quentin Grimes,player_points,PrizePicks,-137,13.5,20.03,OVER,0.91,0.09,"(9.6, 30.2)",UNDER/OVER,0,0.6798,1.039,0.520


## 3 leg parlay

In [12]:
threeLeg = prizepicks3LegEV(
    data=s25,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=4.5,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
279005,Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.8)",Lonzo Ball,player_points,PrizePicks,-137,7.5,12.18,OVER,0.95,0.05,"(6.9, 17.2)",Trey Murphy III,player_points,PrizePicks,-137,19.5,1.70,UNDER,0.05,0.95,"(0.0, 10.9)",UNDER/OVER/UNDER,1,0.8574,4.144,0.829
152768,Deni Avdija,player_points,PrizePicks,-137,19.5,28.56,OVER,0.95,0.05,"(20.1, 37.0)",Jaylon Tyson,player_points,PrizePicks,-137,6.5,18.19,OVER,0.95,0.05,"(8.5, 27.7)",Trey Murphy III,player_points,PrizePicks,-137,19.5,1.70,UNDER,0.05,0.95,"(0.0, 10.9)",OVER/OVER/UNDER,1,0.8574,4.144,0.829
293290,Mikal Bridges,player_points,PrizePicks,-137,15.5,5.20,UNDER,0.05,0.95,"(0.0, 13.2)",Quentin Grimes,player_points,PrizePicks,-137,13.5,20.03,OVER,0.95,0.05,"(14.0, 26.2)",Stephen Curry,player_points,PrizePicks,-137,0.5,29.74,OVER,0.95,0.05,"(12.9, 46.7)",UNDER/OVER/OVER,1,0.8574,4.144,0.829
279240,Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.8)",Mikal Bridges,player_points,PrizePicks,-137,15.5,5.20,UNDER,0.05,0.95,"(0.0, 13.2)",Stephen Curry,player_points,PrizePicks,-137,0.5,29.74,OVER,0.95,0.05,"(12.9, 46.7)",UNDER/UNDER/OVER,1,0.8574,4.144,0.829
288705,Lonzo Ball,player_points,PrizePicks,-137,7.5,12.18,OVER,0.95,0.05,"(6.9, 17.2)",Stephen Curry,player_points,PrizePicks,-137,0.5,29.74,OVER,0.95,0.05,"(12.9, 46.7)",Trey Murphy III,player_points,PrizePicks,-137,19.5,1.70,UNDER,0.05,0.95,"(0.0, 10.9)",OVER/OVER/UNDER,1,0.8574,4.144,0.829
278986,Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.8)",Lonzo Ball,player_points,PrizePicks,-137,7.5,12.18,OVER,0.95,0.05,"(6.9, 17.2)",Quentin Grimes,player_points,PrizePicks,-137,13.5,20.03,OVER,0.95,0.05,"(14.0, 26.2)",UNDER/OVER/OVER,1,0.8574,4.144,0.829
278973,Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.8)",Lonzo Ball,player_points,PrizePicks,-137,7.5,12.18,OVER,0.95,0.05,"(6.9, 17.2)",Mikal Bridges,player_points,PrizePicks,-137,15.5,5.20,UNDER,0.05,0.95,"(0.0, 13.2)",UNDER/OVER/UNDER,1,0.8574,4.144,0.829
278997,Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.8)",Lonzo Ball,player_points,PrizePicks,-137,7.5,12.18,OVER,0.95,0.05,"(6.9, 17.2)",Stephen Curry,player_points,PrizePicks,-137,0.5,29.74,OVER,0.95,0.05,"(12.9, 46.7)",UNDER/OVER/OVER,1,0.8574,4.144,0.829
279229,Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.8)",Mikal Bridges,player_points,PrizePicks,-137,15.5,5.20,UNDER,0.05,0.95,"(0.0, 13.2)",Quentin Grimes,player_points,PrizePicks,-137,13.5,20.03,OVER,0.95,0.05,"(14.0, 26.2)",UNDER/UNDER/OVER,1,0.8574,4.144,0.829
279248,Kentavious Caldwell-Pope,player_points,PrizePicks,-137,9.5,3.84,UNDER,0.05,0.95,"(0.0, 9.8)",Mikal Bridges,player_points,PrizePicks,-137,15.5,5.20,UNDER,0.05,0.95,"(0.0, 13.2)",Trey Murphy III,player_points,PrizePicks,-137,19.5,1.70,UNDER,0.05,0.95,"(0.0, 10.9)",UNDER/UNDER/UNDER,1,0.8574,4.144,0.829
